# Building Decision Trees

## Goal

Build a decision tree from scratch to understand how tree-based machine learning models learn rules, evaluate splits, and make predictions.

The project starts from the simplest possible model and evolves step by step toward numerical splits, deeper trees, and boosting.

---

## Version 1

Build a one-split decision stump using simple categorical rules.

The model will:

- learn a constant prediction
- measure prediction error
- evaluate categorical splits
- select the split with the highest gain
- make binary predictions
- generate a Titanic submission

In [1]:
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

In [3]:
train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Feature engineering

def add_title_feature(df):
    df = df.copy()

    names = df["Name"].str.lower().str.strip()
    df["Title"] = names.str.extract(r" ([a-z]+)\.", expand=False)

    return df

train = add_title_feature(train)
test = add_title_feature(test)

In [5]:
TARGET = "Survived"
FEATURES = ["Age", "Sex", "Title", "Fare", "Pclass"]

In [6]:
# Check missing values before imputation

print("Train missing values:")
print(train[FEATURES].isna().sum())

print("\nTest missing values:")
print(test[FEATURES].isna().sum())

Train missing values:
Age       177
Sex         0
Title       0
Fare        0
Pclass      0
dtype: int64

Test missing values:
Age       86
Sex        0
Title      0
Fare       1
Pclass     0
dtype: int64


In [7]:
# Fill missing numerical values using training medians

age_median = train["Age"].median()
fare_median = train["Fare"].median()

train["Age"] = train["Age"].fillna(age_median)
test["Age"] = test["Age"].fillna(age_median)

test["Fare"] = test["Fare"].fillna(fare_median)

## Data preparation

We create the training features, the target, and the test features used by the model.

In [8]:
# Prepare training and test features

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]

In [9]:
X_train.head()

,Age,Sex,Title,Fare,Pclass
PassengerId,,,,,
1,22.0,male,mr,7.2500,3
2,38.0,female,mrs,71.2833,1
3,26.0,female,miss,7.9250,3
4,35.0,female,mrs,53.1000,1
5,35.0,male,mr,8.0500,3


## Building the model

We now implement the decision stump from scratch, starting from the simplest prediction and gradually adding split evaluation and search.

In [10]:
# Fit a constant predictor

def fit_constant(y):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    return sum(y) / len(y)

In [11]:
# Compute mean squared error

def mean_squared_error(y, prediction):
    if len(y) == 0:
        raise ValueError("Target cannot be empty.")

    total_error = 0

    for value in y:
        total_error += (value - prediction) ** 2

    return total_error / len(y)

In [12]:
# Split target values using a categorical equality rule

def split_categorical(x, y, category):
    y_left = []
    y_right = []

    for feature_value, target_value in zip(x, y):
        if feature_value == category:
            y_left.append(target_value)
        else:
            y_right.append(target_value)

    return y_left, y_right

In [13]:
# Compute the weighted MSE after a categorical split

def categorical_split_mse(x, y, category):
    y_left, y_right = split_categorical(x, y, category)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    left_mse = mean_squared_error(y_left, left_prediction)
    right_mse = mean_squared_error(y_right, right_prediction)

    n_left = len(y_left)
    n_right = len(y_right)

    return (n_left * left_mse + n_right * right_mse) / (n_left + n_right)

In [14]:
# Compute the error reduction produced by a categorical split

def categorical_split_gain(x, y, category):
    base_prediction = fit_constant(y)
    base_mse = mean_squared_error(y, base_prediction)

    split_mse = categorical_split_mse(x, y, category)

    return base_mse - split_mse

In [15]:
# Find the best categorical split for one feature

def find_best_categorical_split(x, y):
    categories = []

    for value in x:
        if value not in categories:
            categories.append(value)

    best_category = None
    best_gain = float("-inf")

    for category in categories:
        gain = categorical_split_gain(x, y, category)

        if gain > best_gain:
            best_category = category
            best_gain = gain

    return best_category, best_gain

In [16]:
# Find the best feature and category combination

def find_best_feature_split(X, y):
    best_feature = None
    best_category = None
    best_gain = float("-inf")

    for feature in X:
        category, gain = find_best_categorical_split(X[feature], y)

        if gain > best_gain:
            best_feature = feature
            best_category = category
            best_gain = gain

    return best_feature, best_category, best_gain

In [17]:
# Train a one-split decision stump

def fit_stump(X, y):
    best_feature, best_category, best_gain = find_best_feature_split(X, y)

    x = X[best_feature]
    y_left, y_right = split_categorical(x, y, best_category)

    left_prediction = fit_constant(y_left)
    right_prediction = fit_constant(y_right)

    return {
        "feature": best_feature,
        "category": best_category,
        "left_prediction": left_prediction,
        "right_prediction": right_prediction,
        "gain": best_gain
    }

In [18]:
# Train the baseline model

model = fit_stump(X_train, y_train)
model

{'feature': 'Title',
 'category': 'mr',
 'left_prediction': 0.15667311411992263,
 'right_prediction': 0.6978609625668449,
 'gain': 0.07133502379453835}

## Prediction and submission

The trained stump generates leaf values, converts them into binary predictions, and creates the Kaggle submission file.

In [19]:
# Predict class probabilities with the trained stump

def predict_proba_stump(X, model):
    predictions = []

    x = X[model["feature"]]

    for value in x:
        if value == model["category"]:
            predictions.append(model["left_prediction"])
        else:
            predictions.append(model["right_prediction"])

    return predictions

In [20]:
# Convert probability predictions into binary classes

def predict_stump(X, model, threshold=0.5):
    probabilities = predict_proba_stump(X, model)
    predictions = []

    for probability in probabilities:
        if probability >= threshold:
            predictions.append(1)
        else:
            predictions.append(0)

    return predictions

In [21]:
# Generate binary predictions for the Kaggle test set

test_predictions = predict_stump(X_test, model)

In [22]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": test_predictions
})

submission.to_csv("submission_v1.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


## Notes

This first version focuses on the basic mechanics of a decision tree.

- The model uses a single split, also called a decision stump.
- Leaf predictions are learned from the target values that reach each branch.
- Split quality is measured using mean squared error and gain.
- Categorical features are evaluated using equality rules.
- Numerical features are still treated with equality rules and are not yet handled as continuous variables.
- Missing numerical values are filled using medians computed from the training set.
- Final Titanic predictions are converted into binary classes (`0` or `1`).

Future versions will introduce new components and gradually evolve the architecture.